# 👗 VÉSTARA x CatVTON Virtual Try-On Server
### High-Fidelity Diffusion Try-On Inference Node in Google Colab

Official Paper: *Concatenation Is All You Need for Virtual Try-On with Diffusion Models* (CVPR 2025)
Official Repository: [https://github.com/Zheng-Chong/CatVTON](https://github.com/Zheng-Chong/CatVTON)
Weights: `zhengchong/CatVTON`

---
### ⚡ Quick Start:
1. In the Colab menu, make sure GPU is selected: **Runtime > Change runtime type > T4 GPU** (or A100/L4).
2. Run **Cell 1** to verify GPU availability.
3. Run **Cell 2** to clone the official repo & install dependencies.
4. Run **Cell 3** to launch the CatVTON inference server.
5. Copy the generated public link (`https://xxxxxxxx.gradio.live`) and paste it into the **VÉSTARA Virtual Try-On page** under *Engine Settings*.

## Step 1: Check GPU Acceleration

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU via Runtime > Change runtime type > T4 GPU")

## Step 2: Clone Official CatVTON Repository & Install Dependencies

In [ ]:
# 1. Clone official repository
!git clone https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON
%cd /content/CatVTON

# 2. Install official repository requirements + API server dependencies
!pip install -q -r requirements.txt
!pip install -q gradio==4.44.1 fastapi uvicorn pydantic python-multipart
!pip install -q accelerate diffusers transformers torchvision

print("✅ CatVTON dependencies installed successfully!")

## Step 3: Launch CatVTON Inference Server (Dual REST API + Gradio UI)

When this cell finishes starting, look for the output line:
```
Running on public URL: https://xxxxxxxxxxxx.gradio.live
```
Copy that URL and paste it into your VÉSTARA web application.

In [ ]:
import os
import sys
import io
import time
import base64
import torch
from PIL import Image
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import gradio as gr

sys.path.append("/content/CatVTON")

from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading AutoMasker...")
try:
    automasker = AutoMasker(densepose_path="DensePose", schp_path="SCHP", device=device)
except Exception as e:
    print("AutoMasker fallback:", e)
    automasker = None

print("Loading official CatVTON Pipeline (zhengchong/CatVTON)...")
pipeline = CatVTONPipeline(
    base_ckpt="zhengchong/CatVTON",
    attn_ckpt="zhengchong/CatVTON",
    attn_ckpt_version="mix",
    weight_dtype=torch_dtype,
    use_tf32=torch.cuda.is_available(),
    device=device
)
print("✅ Model loaded into GPU!")

def decode_base64_to_pil(b64: str) -> Image.Image:
    if "," in b64:
        b64 = b64.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")

def encode_pil_to_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG", quality=95)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

def run_catvton(person_img, garment_img, cloth_type="upper", num_steps=40, guidance_scale=2.5, seed=42):
    if cloth_type not in ["upper", "lower", "overall"]:
        cloth_type = "upper"
    target_size = (768, 1024)
    person_proc = resize_and_crop(person_img, target_size)
    garment_proc = resize_and_padding(garment_img, target_size)
    
    mask = None
    if automasker is not None:
        try:
            mask_res = automasker(person_proc, cloth_type)
            mask = mask_res.get("mask", mask_res) if isinstance(mask_res, dict) else mask_res
        except Exception:
            pass
    if mask is None:
        w, h = target_size
        mask = Image.new("L", (w, h), 0)
        from PIL import ImageDraw
        draw = ImageDraw.Draw(mask)
        if cloth_type == "upper":
            draw.rectangle([int(w*0.15), int(h*0.18), int(w*0.85), int(h*0.60)], fill=255)
        elif cloth_type == "lower":
            draw.rectangle([int(w*0.20), int(h*0.52), int(w*0.80), int(h*0.95)], fill=255)
        else:
            draw.rectangle([int(w*0.15), int(h*0.18), int(w*0.85), int(h*0.95)], fill=255)
            
    gen = torch.Generator(device=device).manual_seed(seed if seed >= 0 else int(time.time()))
    out = pipeline(
        image=person_proc,
        condition_image=garment_proc,
        mask=mask,
        num_inference_steps=int(num_steps),
        guidance_scale=float(guidance_scale),
        generator=gen
    )
    return out[0]

with gr.Blocks(title="VÉSTARA CatVTON Node") as demo:
    gr.Markdown("# 👗 VÉSTARA x CatVTON Inference Node\nPaste public link into VÉSTARA.")
    with gr.Row():
        with gr.Column():
            p_in = gr.Image(type="pil", label="Person")
            g_in = gr.Image(type="pil", label="Garment")
            c_type = gr.Radio(["upper", "lower", "overall"], value="upper", label="Type")
            b_run = gr.Button("Generate", variant="primary")
        with gr.Column():
            p_out = gr.Image(type="pil", label="Result")
    b_run.click(run_catvton, inputs=[p_in, g_in, c_type], outputs=[p_out], api_name="tryon")

app: FastAPI = demo.app
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
async def health():
    return {"status": "ok", "model": "CatVTON (Diffusion Model)", "cuda": torch.cuda.is_available()}

@app.post("/api/tryon")
async def api_tryon(request: Request):
    body = await request.json()
    p_pil = decode_base64_to_pil(body["person_image"])
    g_pil = decode_base64_to_pil(body["garment_image"])
    res = run_catvton(
        person_img=p_pil,
        garment_img=g_pil,
        cloth_type=body.get("cloth_type", "upper"),
        num_steps=int(body.get("num_inference_steps", 40)),
        guidance_scale=float(body.get("guidance_scale", 2.5)),
        seed=int(body.get("seed", 42))
    )
    return {"status": "success", "result_image": encode_pil_to_base64(res), "model": "CatVTON (Diffusion Model)"}

demo.queue().launch(share=True, server_name="0.0.0.0", server_port=7860)
